# Using NequIP in DeepChem

Machine-learned interatomic potentials (MLIPs) use atomic species and geometry to predict structure-level quantities such as total energy and atom-level quantities such as forces. NequIP is an E(3)-equivariant architecture: its internal features respect rotations, translations, and reflections of the atomic structure.

This short tutorial builds a tiny ASE-readable dataset, converts its structures to DeepChem radius graphs, batches structures with different numbers of atoms, and runs energy and force inference with DeepChem's native NequIP backbone. No dataset download or training is involved.

## Setup

Run this notebook from a DeepChem source environment containing the NequIP backbone (PR #5109), with PyTorch and ASE installed. Because that backbone is not yet in a released DeepChem package, installing DeepChem from PyPI inside this notebook would not provide the API used below.

In [ ]:
from pathlib import Path
import tempfile

import numpy as np
import torch
from ase import Atoms
from ase.io import write

import deepchem as dc
from deepchem.feat.graph_data import BatchGraphData
from deepchem.models.torch_models import NequIP

## Create a tiny ASE-readable dataset

ASE's extended XYZ format can store a sequence of structures, per-structure metadata, and per-atom arrays. Here the energy and force values are deliberately synthetic; they only demonstrate how labels flow through `MaterialsLoader`. The custom key names avoid any calculator-specific handling of the standard ASE `energy` and `forces` names. DeepChem preserves the supplied numbers and does not infer or convert their units.

In [ ]:
h2 = Atoms("H2", positions=[[0.0, 0.0, 0.0], [0.74, 0.0, 0.0]])
h2.info["reference_energy"] = -1.0
h2.new_array("reference_forces", np.array([[0.05, 0.0, 0.0],
                                                    [-0.05, 0.0, 0.0]], dtype=np.float32))

water = Atoms("OH2", positions=[[0.0000, 0.0000, 0.0000],
                                      [0.9572, 0.0000, 0.0000],
                                      [-0.2390, 0.9270, 0.0000]])
water.info["reference_energy"] = -2.0
water.new_array("reference_forces", np.array([[0.00, 0.02, 0.0],
                                                       [0.01, -0.01, 0.0],
                                                       [-0.01, -0.01, 0.0]], dtype=np.float32))

structures = [h2, water]
temporary_directory = tempfile.TemporaryDirectory()
xyz_path = Path(temporary_directory.name) / "tiny_structures.extxyz"
write(xyz_path, structures, format="extxyz")
print(f"Wrote {len(structures)} structures to {xyz_path.name}")

## Featurize and load the structures

`AtomisticRadiusGraphFeaturizer` creates a non-periodic, directed edge for every atom pair separated by less than `cutoff`. The NequIP input topology must use the same cutoff as the model. `MaterialsLoader` reads each ASE frame, applies this featurizer, and places the requested labels in a DeepChem `DiskDataset`. Force labels remain ragged because the structures can contain different numbers of atoms.

In [ ]:
cutoff = 2.5
featurizer = dc.feat.AtomisticRadiusGraphFeaturizer(cutoff=cutoff)
preview_graph = featurizer.featurize([structures[0]])[0]
print(preview_graph)

In [ ]:
loader = dc.data.MaterialsLoader(
    tasks=["energy", "forces"],
    featurizer=featurizer,
    energy_key="reference_energy",
    forces_key="reference_forces",
)
dataset = loader.create_dataset(
    str(xyz_path), data_dir=str(Path(temporary_directory.name) / "dataset")
)

print("Tasks:", dataset.get_task_names())
print("Energy labels:", [float(value) for value in dataset.y[:, 0]])
print("Force-label shapes:", [value.shape for value in dataset.y[:, 1]])

## Inspect a `GraphData` object

`node_features` contains atomic numbers and `node_pos_features` contains Cartesian positions. `edge_index[0]` and `edge_index[1]` are the source and destination atom indices. For each directed source-to-destination edge, `edge_features` stores the displacement `position[destination] - position[source]`, while `edge_distances` stores its length. NequIP uses the topology but recomputes these geometric quantities from positions so that force gradients remain connected to the coordinates.

In [ ]:
graph = dataset.X[0]
print(graph)
print("Atomic numbers:", graph.node_features[:, 0].tolist())
print("Position shape:", graph.node_pos_features.shape)
print("Edge-index shape:", graph.edge_index.shape)
print("Edge-vector shape:", graph.edge_features.shape)
print("Edge distances:", graph.edge_distances[:, 0].round(3).tolist())

## Batch variable-size structures

`BatchGraphData` concatenates node and edge arrays, offsets the edge indices, and adds `graph_index` to map each atom back to its structure. `numpy_to_torch()` performs the tensor conversion required by NequIP.

In [ ]:
batch = BatchGraphData(list(dataset.X)).numpy_to_torch()
print("Batched node shape:", tuple(batch.node_features.shape))
print("Batched edge-index shape:", tuple(batch.edge_index.shape))
print("Structure membership:", batch.graph_index.tolist())

## Run the native NequIP backbone

The following deliberately small configuration keeps this demonstration fast. The model is randomly initialized: its outputs are useful for checking shapes and API flow, but they are **not scientifically meaningful predictions**.

In [ ]:
torch.manual_seed(7)
model = NequIP(
    cutoff=cutoff,
    num_species=10,  # Must exceed the largest atomic number (oxygen is 8).
    hidden_channels=2,
    num_interaction_blocks=2,
    l_max=1,
    num_bessel=4,
    radial_hidden_width=8,
    radial_hidden_depth=1,
    avg_num_neighbors=2.0,
    readout_hidden_channels=4,
)
model.eval()

with torch.no_grad():
    predicted_energies = model(batch)
print("Energy-output shape:", tuple(predicted_energies.shape))
print(predicted_energies)

Pass `compute_forces=True` to return both total structure energies and one force vector per atom. These forces are conservative by construction because the backbone obtains them from the same scalar energy as $\mathbf{F}_i = -\partial E/\partial \mathbf{R}_i$.

In [ ]:
energies_with_grad, predicted_forces = model(batch, compute_forces=True)
print("Energy-output shape:", tuple(energies_with_grad.shape))
print("Force-output shape:", tuple(predicted_forces.shape))
print(predicted_forces)

## Current scope

PR #5109 provides the native NequIP **backbone only**. It does not yet provide a DeepChem `TorchModel` training wrapper or an end-to-end `fit()` workflow. Consequently, this tutorial intentionally stops at direct energy and force inference and does not present training, evaluation, or checkpointing. A future wrapper can connect labeled `MaterialsLoader` datasets like this one to DeepChem's training APIs.